# Lesson 5: Set and verify power in standby

Use raw `read()` and `write()` exchanges to change only the stored setpoint.
The laser must already be in **STANDBY**, the shutter **CLOSED**, with no active
faults. This lesson checks these conditions before and after the write; it never
sends laser-enable, shutter-open, standby or shutter-close commands.

Complete [pre-key-on readiness](04_pre_key_on_readiness.ipynb) and follow the
operator-approved procedure. The setpoint remains stored after disconnecting.
Run cells in order with one serial owner and the instrument prompt enabled.
Each call waits for the complete reply and prompt. Replies remain raw bytes.

**Stop after any error.** A write may already have executed. Inspect the saved
transaction, disconnect and resolve the cause; do not retry or send blind cleanup.
Software checks cannot prevent someone changing the physical controls between
queries and cannot establish physical beam safety.

## 1. Connection and approved power settings

Confirm port, model and baud with the operator. Enter the independently verified
no-active-fault reply as **bytes**, for example `b"SYSTEM OK"` only if verified for
this instrument. Do not infer it from fault history or an empty response.
Set `TARGET_W` and the site-approved `POWER_LIMIT_W` in watts; no default power
is supplied. Restart the kernel after package updates.

In [ ]:
PORT = "<PORT>"
MODEL = "<MODEL>"  # V2, V5 or V6
BAUDRATE = 19200
TIMEOUT_S = 2.0
ACTIVE_FAULT_CLEAR_REPLY = None  # Verified payload bytes, not a full wire reply
TARGET_W = None  # Operator-approved setpoint in watts
POWER_LIMIT_W = None  # Site-approved ceiling in watts

## 2. Validate settings before opening the port

The instruction is built directly with byte formatting: `b"P=%.4f"`.
Both the original target and the four-decimal transmitted value must be within
the ceiling. Construction sends nothing. Writes require explicit enablement.

In [ ]:
from math import isfinite

from coherent_verdi import VerdiController

if PORT == "<PORT>" or POWER_LIMIT_W is None or ACTIVE_FAULT_CLEAR_REPLY is None:
    raise ValueError("Fill in every setting in section 1 first.")
if (
    not isinstance(ACTIVE_FAULT_CLEAR_REPLY, bytes)
    or not ACTIVE_FAULT_CLEAR_REPLY
    or ACTIVE_FAULT_CLEAR_REPLY.strip(b" ") != ACTIVE_FAULT_CLEAR_REPLY
    or any(c < 32 or c > 126 for c in ACTIVE_FAULT_CLEAR_REPLY)
    or b"&" in ACTIVE_FAULT_CLEAR_REPLY
    or (ACTIVE_FAULT_CLEAR_REPLY.isdigit() and int(ACTIVE_FAULT_CLEAR_REPLY) > 0)
):
    raise ValueError("ACTIVE_FAULT_CLEAR_REPLY must be verified nonempty payload bytes.")
laser = VerdiController(
    PORT,
    model=MODEL,
    baudrate=BAUDRATE,
    timeout_s=TIMEOUT_S,
    prompt_enabled=True,
    allow_writes=True,
    power_limit_w=POWER_LIMIT_W,
)
if type(TARGET_W) not in (int, float) or not isfinite(TARGET_W):
    raise ValueError("TARGET_W must be a finite number.")
if not 0 <= TARGET_W <= laser.power_limit_w:
    raise ValueError("TARGET_W must be between 0 and POWER_LIMIT_W.")
target_bytes = b"%.4f" % TARGET_W
sent_w = float(target_bytes)
if not 0 <= sent_w <= laser.power_limit_w:
    raise ValueError("Rounded TARGET_W exceeds POWER_LIMIT_W.")
power_command = b"P=" + target_bytes
print("Will send:", repr(power_command))

## 3. Compare byte payloads explicitly

Raw replies stay unchanged in the notebook and transaction record. This small
helper checks the known terminal framing and removes a recognized prompt and
exact instruction echo **only for the comparisons below**. It never encodes or
decodes text and does not attempt to repair interleaved or corrupted replies.

In [ ]:
prompts = (b"Verdi>", {"V2": b"Verdi V-2>", "V5": b"Verdi V-5>", "V6": b"Verdi V-6>"}[MODEL])


def reply_payload(instruction, wire):
    for prompt in prompts:
        if wire.endswith(b"\r\n" + prompt):
            body = wire[: -(len(prompt) + 2)]
            break
    else:
        raise RuntimeError(f"Unexpected reply framing: {wire!r}")
    for prompt in prompts:
        if body.startswith(prompt):
            body = body[len(prompt) :].lstrip(b" ")
            break
    body = body.removeprefix(instruction).strip(b" ")
    if b"\r" in body or b"\n" in body:
        raise RuntimeError(f"Unexpected extra reply data: {wire!r}")
    return body

## 4. Open the serial port

Run this cell by itself and inspect the result before starting serial communication.
`connect()` opens the local serial port and returns `True` on success. Failures
raise an exception. It sends no instrument instructions. A successful open does
not establish that the laser is responding or ready.

Close any other serial owner first. If opening fails, stop here.

`override=True` closes an older controller connection to the same port in this
Python process before opening the new one. It cannot take ownership from another
kernel/application or a raw `ser` handle; release those explicitly. A failed
controller still requires a fresh controller object. Reopening does not establish
instrument synchronization or repair corrupted replies.

In [ ]:
connected = laser.connect(override=True)
print("Serial port open:", connected)

## 5. Start serial communication

Only after inspecting the connection result, run the following cells separately.
Buffer maintenance and instrument queries belong here, outside the connection
section. An explicit purge discards host queues; it does not reset the device's
parser. No delay or purge is claimed to prove synchronization. Each query waits
for its complete reply and prompt before another can be sent.

In [ ]:
laser.purge_buffers()
version_reply = laser.read(b"?SV")
print("Version:", repr(version_reply))

before = {}
for query in (b"?L", b"?S", b"?F", b"?SP"):
    before[query] = laser.read(query)
    print(query, repr(before[query]))
if before[b"?L"] != b"0" or before[b"?S"] != b"0" or before[b"?F"] != ACTIVE_FAULT_CLEAR_REPLY:
    raise RuntimeError("Require STANDBY, shutter closed and no active faults.")

## 6. Recheck and send the setpoint once

Type `SET` to proceed. Recheck immediately afterward because conditions may have
changed during confirmation. Only `P=` is sent. The complete acknowledgment is
saved and printed; an unexpected payload stops the lesson before verification.

In [ ]:
if input("To store the new setpoint, type SET: ").strip() != "SET":
    raise RuntimeError("Cancelled. No settings were changed.")

now = {}
for query in (b"?L", b"?S", b"?F"):
    now[query] = laser.read(query)
    print(query, repr(now[query]))
if now[b"?L"] != b"0" or now[b"?S"] != b"0" or now[b"?F"] != ACTIVE_FAULT_CLEAR_REPLY:
    raise RuntimeError("Require STANDBY, shutter closed and no active faults.")

acknowledgment = laser.write(power_command)
print("Acknowledgment:", repr(acknowledgment))
if reply_payload(power_command, acknowledgment) != b"":
    raise RuntimeError(f"Unexpected command acknowledgment: {acknowledgment!r}")

## 7. Verify stored power and unchanged state

`?SP` reports the stored setpoint, not emitted optical power. This lesson requires
the expected four-decimal payload; a different precision or value stops verification
rather than guessing a conversion or tolerance. Check standby, shutter and faults
again. These sequential checks establish reported state, not a continuous interlock.

In [ ]:
after = {}
for query in (b"?SP", b"?L", b"?S", b"?F"):
    after[query] = laser.read(query)
    print(query, repr(after[query]))
if after[b"?SP"] != target_bytes:
    raise RuntimeError(f"Setpoint readback mismatch: {after[b'?SP']!r}")
if after[b"?L"] != b"0" or after[b"?S"] != b"0" or after[b"?F"] != ACTIVE_FAULT_CLEAR_REPLY:
    raise RuntimeError("State or faults changed unexpectedly after the write.")
print("Verified setpoint payload:", repr(target_bytes), "; standby and shutter closed reported.")

## 8. Inspect an error

Print the saved transaction without sending anything. If the write was attempted,
its outcome may be uncertain; do not repeat it or send cleanup commands.

In [ ]:
print(laser.last_transaction)

## 9. Disconnect

Releases the port without changing laser state. The setpoint remains stored.
This lesson issues no emission or shutter-control commands.

In [ ]:
laser.disconnect()